# Fitted Nominal, Real, and Breakeven Curves

Fit tiny synthetic nominal and TIPS bond panels in price space, keep the fit in dirty-price space internally, use no external regressors, derive direct par curves from the fitted curves, and build zero breakeven plus par breakeven as dedicated curve objects.


In [ ]:
import importlib
import os
import warnings
from decimal import Decimal

os.makedirs('/tmp/matplotlib', exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
warnings.filterwarnings(
    'ignore',
    message='invalid value encountered in matmul',
    category=RuntimeWarning,
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fuggers_py.core import Currency, Date, Frequency, YearMonth
from fuggers_py.market.snapshot import InflationFixing
from fuggers_py.market.sources import InMemoryInflationFixingSource
import fuggers_py.market.quotes as market_quotes
import fuggers_py.market.curves.fitted_bonds.fair_value as fitted_fair_value
import fuggers_py.market.curves.fitted_bonds.model as fitted_model
import fuggers_py.market.curves.fitted_bonds.optimization as fitted_optimization
import fuggers_py.market.curves.fitted_bonds.par_curve as fitted_par_curve
import fuggers_py.market.curves.fitted_bonds.pricing_adapters as fitted_pricing_adapters
import fuggers_py.market.curves.inflation.breakeven as inflation_breakeven

for module in (
    market_quotes,
    fitted_fair_value,
    fitted_model,
    fitted_pricing_adapters,
    fitted_optimization,
    fitted_par_curve,
    inflation_breakeven,
):
    importlib.reload(module)

BondQuote = market_quotes.BondQuote
BreakevenParCurve = inflation_breakeven.BreakevenParCurve
BreakevenZeroCurve = inflation_breakeven.BreakevenZeroCurve
CubicSplineZeroRateCurveModel = fitted_model.CubicSplineZeroRateCurveModel
FittedBondCurveFitter = fitted_optimization.FittedBondCurveFitter
FittedParYieldCurve = fitted_par_curve.FittedParYieldCurve
NominalGovernmentBondPricingAdapter = fitted_pricing_adapters.NominalGovernmentBondPricingAdapter
ParCurveSpec = fitted_par_curve.ParCurveSpec
TipsRealBondPricingAdapter = fitted_pricing_adapters.TipsRealBondPricingAdapter
dirty_price_from_curve = fitted_fair_value.dirty_price_from_curve
from fuggers_py.products.bonds import FixedBondBuilder, TipsBond
from fuggers_py.reference.bonds.types import YieldCalculationRules
from fuggers_py.reference.inflation import USD_CPI_U_NSA


## 1. Setup

Fix one reference date, one spline grid, one deterministic CPI path, and two smooth synthetic curves that generate the example prices.


In [ ]:
REFERENCE_DATE = Date.from_ymd(2026, 1, 15)
KNOT_TENORS = (Decimal('2.0'), Decimal('4.0'), Decimal('7.0'), Decimal('10.0'))
CURVE_MODEL = CubicSplineZeroRateCurveModel(KNOT_TENORS)
TRUE_NOMINAL_CURVE = CURVE_MODEL.build_curve(
    REFERENCE_DATE,
    np.asarray([0.024, 0.027, 0.031, 0.034], dtype=float),
    max_t=10.0,
)
TRUE_REAL_CURVE = CURVE_MODEL.build_curve(
    REFERENCE_DATE,
    np.asarray([-0.003, 0.000, 0.004, 0.008], dtype=float),
    max_t=10.0,
)

def make_fixing_source() -> InMemoryInflationFixingSource:
    fixings: list[InflationFixing] = []
    level = Decimal('100.00')
    year = 2023
    month = 10
    while (year, month) <= (2036, 12):
        fixings.append(
            InflationFixing(
                index_name='CPURNSA',
                observation_month=YearMonth(year=year, month=month),
                value=level,
            )
        )
        level += Decimal('0.22')
        month += 1
        if month > 12:
            month = 1
            year += 1
    return InMemoryInflationFixingSource(fixings)

FIXING_SOURCE = make_fixing_source()

def build_nominal_bond(maturity_years: int, coupon_rate: Decimal):
    return (
        FixedBondBuilder.new()
        .with_issue_date(Date.from_ymd(2021, 1, 15))
        .with_maturity_date(REFERENCE_DATE.add_years(maturity_years))
        .with_coupon_rate(coupon_rate)
        .with_frequency(Frequency.SEMI_ANNUAL)
        .with_currency(Currency.USD)
        .with_rules(YieldCalculationRules.us_treasury())
        .build()
    )

def build_tips_bond(maturity_years: int, coupon_rate: Decimal) -> TipsBond:
    return TipsBond.new(
        issue_date=Date.from_ymd(2024, 1, 15),
        dated_date=Date.from_ymd(2024, 1, 15),
        maturity_date=REFERENCE_DATE.add_years(maturity_years),
        coupon_rate=coupon_rate,
        inflation_convention=USD_CPI_U_NSA,
        base_reference_date=Date.from_ymd(2024, 1, 15),
        frequency=Frequency.SEMI_ANNUAL,
        currency=Currency.USD,
    )

def tips_dirty_price_from_curve(bond: TipsBond) -> Decimal:
    present_value = Decimal('0')
    for cash_flow in bond.projected_cash_flows(
        fixing_source=FIXING_SOURCE,
        settlement_date=REFERENCE_DATE,
    ):
        present_value += cash_flow.factored_amount() * TRUE_REAL_CURVE.discount_factor(cash_flow.date)
    return present_value


## 2. Synthetic Observations

Build one small nominal cross-section and one small TIPS cross-section directly from the synthetic curves.


In [ ]:
nominal_inputs = [
    ('UST2Y', 2, Decimal('0.0200')),
    ('UST3Y', 3, Decimal('0.0225')),
    ('UST4Y', 4, Decimal('0.0250')),
    ('UST5Y', 5, Decimal('0.0275')),
    ('UST7Y', 7, Decimal('0.0300')),
    ('UST10Y', 10, Decimal('0.0350')),
]

nominal_quotes = []
nominal_bonds = {}
nominal_rows = []
for instrument_id, maturity_years, coupon_rate in nominal_inputs:
    bond = build_nominal_bond(maturity_years, coupon_rate)
    nominal_bonds[instrument_id] = bond
    clean_price = dirty_price_from_curve(bond, TRUE_NOMINAL_CURVE, REFERENCE_DATE) - bond.accrued_interest(REFERENCE_DATE)
    nominal_quotes.append(
        BondQuote(
            instrument_id=instrument_id,
            clean_price=clean_price,
            as_of=REFERENCE_DATE,
            currency=Currency.USD,
        )
    )
    nominal_rows.append(
        {
            'instrument_id': instrument_id,
            'maturity_years': maturity_years,
            'coupon_pct': 100 * float(coupon_rate),
            'clean_price': float(clean_price),
        }
    )

nominal_quote_table = pd.DataFrame(nominal_rows).round(3)
nominal_quote_table


In [ ]:
tips_inputs = [
    ('TIPS2Y', 2, Decimal('0.0050')),
    ('TIPS3Y', 3, Decimal('0.0060')),
    ('TIPS5Y', 5, Decimal('0.0075')),
    ('TIPS7Y', 7, Decimal('0.0090')),
    ('TIPS10Y', 10, Decimal('0.0110')),
]

tips_quotes = []
tips_bonds = {}
tips_rows = []
for instrument_id, maturity_years, coupon_rate in tips_inputs:
    bond = build_tips_bond(maturity_years, coupon_rate)
    tips_bonds[instrument_id] = bond
    clean_price = tips_dirty_price_from_curve(bond) - bond.accrued_interest(REFERENCE_DATE, fixing_source=FIXING_SOURCE)
    tips_quotes.append(
        BondQuote(
            instrument_id=instrument_id,
            clean_price=clean_price,
            as_of=REFERENCE_DATE,
            currency=Currency.USD,
        )
    )
    tips_rows.append(
        {
            'instrument_id': instrument_id,
            'maturity_years': maturity_years,
            'coupon_pct': 100 * float(coupon_rate),
            'real_clean_price': float(clean_price),
        }
    )

tips_quote_table = pd.DataFrame(tips_rows).round(3)
tips_quote_table


## 3. Fit The Curves

Run the same generic fitted-bond engine for nominal and real curves. Both fits use the cubic zero-rate spline and no regressors.


In [ ]:
nominal_fit = FittedBondCurveFitter(
    curve_model=CURVE_MODEL,
    pricing_adapter=NominalGovernmentBondPricingAdapter(),
).fit(
    nominal_quotes,
    bonds=nominal_bonds,
    settlement_date=REFERENCE_DATE,
    regression_exposures={},
)

real_fit = FittedBondCurveFitter(
    curve_model=CURVE_MODEL,
    pricing_adapter=TipsRealBondPricingAdapter(FIXING_SOURCE),
).fit(
    tips_quotes,
    bonds=tips_bonds,
    settlement_date=REFERENCE_DATE,
    regression_exposures={},
)

fit_diagnostics = pd.DataFrame(
    [
        {
            'curve': 'nominal',
            'converged': nominal_fit.diagnostics.converged,
            'weighted_rmse_price': float(nominal_fit.diagnostics.weighted_rmse_price),
            'max_abs_bp_residual': float(nominal_fit.diagnostics.max_abs_bp_residual),
        },
        {
            'curve': 'real',
            'converged': real_fit.diagnostics.converged,
            'weighted_rmse_price': float(real_fit.diagnostics.weighted_rmse_price),
            'max_abs_bp_residual': float(real_fit.diagnostics.max_abs_bp_residual),
        },
    ]
).round(8)
fit_diagnostics


## 4. Par And Breakeven Views

Derive the nominal and real par curves from the fitted curves, then build zero breakeven and par breakeven on top.


In [ ]:
par_spec = ParCurveSpec(frequency=Frequency.SEMI_ANNUAL, currency=Currency.USD)
nominal_par_curve = FittedParYieldCurve.from_fit_result(nominal_fit, par_spec)
real_par_curve = FittedParYieldCurve.from_fit_result(real_fit, par_spec)
zero_breakeven_curve = BreakevenZeroCurve.from_fitted_curves(nominal_fit, real_fit)
par_breakeven_curve = BreakevenParCurve.from_par_curves(nominal_par_curve, real_par_curve)

sample_tenors = [Decimal('2'), Decimal('3'), Decimal('5'), Decimal('7'), Decimal('10')]
summary = pd.DataFrame(
    {
        'tenor_years': [float(tenor) for tenor in sample_tenors],
        'nominal_zero_pct': [100 * nominal_fit.curve.zero_rate_at_tenor(float(tenor)) for tenor in sample_tenors],
        'real_zero_pct': [100 * real_fit.curve.zero_rate_at_tenor(float(tenor)) for tenor in sample_tenors],
        'zero_breakeven_pct': [100 * float(zero_breakeven_curve.zero_breakeven(tenor)) for tenor in sample_tenors],
        'nominal_par_pct': [100 * float(nominal_par_curve.par_yield(tenor)) for tenor in sample_tenors],
        'real_par_pct': [100 * float(real_par_curve.par_yield(tenor)) for tenor in sample_tenors],
        'par_breakeven_pct': [100 * float(par_breakeven_curve.par_breakeven(tenor)) for tenor in sample_tenors],
    }
).round(3)
summary


In [ ]:
dense_tenors = np.linspace(0.5, 10.0, 60)

figure, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

axes[0, 0].plot(
    dense_tenors,
    [100 * nominal_fit.curve.zero_rate_at_tenor(float(tenor)) for tenor in dense_tenors],
)
axes[0, 0].set_title('Nominal zero curve')
axes[0, 0].set_xlabel('Tenor (years)')
axes[0, 0].set_ylabel('Rate (%)')
axes[0, 0].grid(True)

axes[0, 1].plot(
    dense_tenors,
    [100 * real_fit.curve.zero_rate_at_tenor(float(tenor)) for tenor in dense_tenors],
)
axes[0, 1].set_title('Real zero curve')
axes[0, 1].set_xlabel('Tenor (years)')
axes[0, 1].set_ylabel('Rate (%)')
axes[0, 1].grid(True)

axes[1, 0].plot(
    dense_tenors,
    [100 * float(zero_breakeven_curve.zero_breakeven(Decimal(str(float(tenor))))) for tenor in dense_tenors],
)
axes[1, 0].set_title('Zero breakeven curve')
axes[1, 0].set_xlabel('Tenor (years)')
axes[1, 0].set_ylabel('Rate (%)')
axes[1, 0].grid(True)

axes[1, 1].plot(
    summary['tenor_years'],
    summary['par_breakeven_pct'],
    marker='o',
)
axes[1, 1].set_title('Par breakeven samples')
axes[1, 1].set_xlabel('Tenor (years)')
axes[1, 1].set_ylabel('Rate (%)')
axes[1, 1].grid(True)

plt.show()
